In [ ]:
"""
session_summary.ipynb

plot model metrics across sessions

Author: Stellina X. Ao
Created: 2026-05-04
Last Modified: 2026-05-06
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import pickle
import numpy as np
import matplotlib.pyplot as plt
from utils.paths import FIGURES_DIR, MODELS_DIR

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

## LVM - one latent

In [ ]:
"""
ALL & PER-REGION
R2 of encoding and latent variable model 
Improvement over task variable model and QI
Single latent histograms
Quantify time scale of latents (PSD)
"""

In [ ]:
from core.data import subject_ids, session_ids, colors_subj


def get_res_tv_lvms(subj_id, regions=["all", "ACC", "M2", "DMS", "DLS"]):
    subj_idx = np.where(subject_ids == subj_id)[0][0]

    res_tv_lvms = []
    for sess_id in session_ids[subj_idx]:
        file_path = (
            MODELS_DIR / "fit" / subj_id / sess_id / "one_latent" / "results_dict.pkl"
        )

        with open(file_path, "rb") as f:
            res_dict = pickle.load(f)

        res_tv_lvm = {}
        for region in regions:
            try:
                res_tv_lvms_ = res_dict[region]["res_tv_lvms"]
            except KeyError:
                continue
            # for now
            if len(res_tv_lvms_) == 0:
                continue
            # print(subj_id, sess_id, region)
            res_tv_lvm[region] = {
                key: np.array([res_tv_lvm_[key] for res_tv_lvm_ in res_tv_lvms_])
                for key in res_tv_lvms_[0].keys()
            }
        res_tv_lvms.append(res_tv_lvm)

    return res_tv_lvms

In [ ]:
res_tv_lvms = {
    subj_id: get_res_tv_lvms(subj_id) for subj_id in ["MM012", "MR82", "MR83"]
}

In [ ]:
region = "ACC"
metric = "r2test_diff"

fig, ax = plt.subplots()
for subj_id in ["MM012", "MR82", "MR83"]:
    metrics_avg = np.array(
        [
            res_tv_lvm[region][metric].mean() if region in res_tv_lvm.keys() else np.nan
            for res_tv_lvm in res_tv_lvms[subj_id]
        ]
    )
    metrics_std = np.array(
        [
            res_tv_lvm[region][metric].std() if region in res_tv_lvm.keys() else np.nan
            for res_tv_lvm in res_tv_lvms[subj_id]
        ]
    )

    ax.plot(metrics_avg, color=colors_subj[subj_id], label=subj_id)
    ax.fill_between(
        np.arange(len(metrics_avg)),
        metrics_avg - metrics_std,
        metrics_avg + metrics_std,
        color=colors_subj[subj_id],
        alpha=0.25,
    )

ax.axhline(y=0, color="#666666", linestyle="--")
ax.set_xlabel("session")
ax.set_ylabel(metric)
ax.legend()
fig.tight_layout()


save_dir = FIGURES_DIR / "session_summary"
fpath = save_dir / f"{metric}_{region}.png"
fpath.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(fpath, dpi=300, bbox_inches="tight")

## LVM - tributaries

In [ ]:
def load_family_rt(subj_id, sess_id, region, epoch, seed):
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "river_n_tributaries"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        return

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)

    try:
        family = res_dict[region][epoch["alignment"]]["both"]["families"][seed]
        family_mb = res_dict[region][epoch["alignment"]]["mb"]["families"][seed]
        family_mf = res_dict[region][epoch["alignment"]]["mf"]["families"][seed]
    except IndexError:
        return

    return family, family_mb, family_mf

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"
region = "all"
epoch = {"alignment": "choice", "tpre": 0.5, "tpost": 0.5}
seed = 2

family, family_mb, family_mf = load_family_rt(subj_id, sess_id, region, epoch, seed)

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

file_path = (
    MODELS_DIR / "fit" / subj_id / sess_id / "river_n_tributaries" / "results_dict.pkl"
)

with open(file_path, "rb") as f:
    res_dict = pickle.load(f)

In [ ]:
# plot across session p(r choice | mb)nd p(r|mf) and rewarded and unrewarded

In [ ]:
# fit full model for choice + reward epoch first then use that as the filtering criteria
# > solution is to add a period for filtering in the get_psths function <-- check that num_units is = across epochs
# with the full balanced epoch model find cids and use those to update what the lvm filters out

In [ ]:
# do action item 2 and confirm unit count and cids match for the three row models
# > check check check <- number of neurons across epochs for tvm is equal
# > run for one sess
# > load into notebook and check check check
# > launch for everything

In [ ]:
epochs = [
    {"key": "choice", "alignment": "choice", "tpre": 0.5, "tpost": 0.5},
    {"key": "reward", "alignment": "reward", "tpre": 0, "tpost": 1},
    {"key": "iti", "alignment": "trial_start", "tpre": 1.5, "tpost": -0.5},
]

epochs_key = [epoch["key"] for epoch in epochs]

In [ ]:
from sg.fitter import LVMFamily

subj_id = "MR82"
sess_id = "20251027_152036"
epoch = epochs[0]

epoch_ref = {"alignment": "choice", "tpre": 0.5, "tpost": 1}
seed_sample = 0
best_regl_consts = (0.01, 0.01)
region = "all"

family = LVMFamily(
    subj_id=subj_id,
    sess_id=sess_id,
    n_latents_mult=1,
    n_latents_addt=1,
    regions=None if region == "all" else [region],
    refit=False,
    alignment=epoch["alignment"],
    tpre=epoch["tpre"],
    tpost=epoch["tpost"],
    alignment_ref=epoch_ref["alignment"],
    tpre_ref=epoch_ref["tpre"],
    tpost_ref=epoch_ref["tpost"],
    n_splines=2,
    balance_strategy=True,
    seed=1234,
    tv_reg={"l2": best_regl_consts[0]},
    reg={"l2": best_regl_consts[1]},
)
family.get_data()
family.fit_baseline()
family.fit_taskvar()
family.get_cids()
family.update_cids()
family.eval()

family_mb = LVMFamily(
    subj_id=subj_id,
    sess_id=sess_id,
    n_latents_mult=1,
    n_latents_addt=1,
    regions=None if region == "all" else [region],
    refit=False,
    idxs_subsamp=family.idxs_subsamp_mb,
    alignment=epoch["alignment"],
    tpre=epoch["tpre"],
    tpost=epoch["tpost"],
    alignment_ref=epoch_ref["alignment"],
    tpre_ref=epoch_ref["tpre"],
    tpost_ref=epoch_ref["tpost"],
    binwidth_ms=25,
    n_splines=2,
    seed=1234,
    tv_reg={"l2": best_regl_consts[0]},
    reg={"l2": best_regl_consts[1]},
)
family_mb.fit_all(family.cids)
family_mb.eval()

family_mf = LVMFamily(
    subj_id=subj_id,
    sess_id=sess_id,
    n_latents_mult=1,
    n_latents_addt=1,
    regions=None if region == "all" else [region],
    refit=False,
    idxs_subsamp=family.idxs_subsamp_mf,
    alignment=epoch["alignment"],
    tpre=epoch["tpre"],
    tpost=epoch["tpost"],
    alignment_ref=epoch_ref["alignment"],
    tpre_ref=epoch_ref["tpre"],
    tpost_ref=epoch_ref["tpost"],
    binwidth_ms=25,
    n_splines=2,
    seed=1234,
    tv_reg={"l2": best_regl_consts[0]},
    reg={"l2": best_regl_consts[1]},
)
family_mf.fit_all(family.cids)
family_mf.eval()

In [ ]:
family.num_units, family_mb.num_units

In [ ]:
family.num_trials, family_mb.num_trials

In [ ]:
from utils.paths import FIGURES_DIR
from squiggs.neuron_viewer import NeuronViewer
from squiggs.renderers import FitRenderer
from pathlib import Path

family = family_mf

renderer = FitRenderer(
    family.mod_taskvar,
    x=family.test_dl.dataset[:],
    y=family.robs,
    dfs=family.test_dl.dataset[:]["dfs"][:, family.cids].detach().cpu().numpy(),
    save_subdir=Path("model_fits") / subj_id / sess_id / "taskvar",
)

nv = NeuronViewer(
    num_units=renderer.y.shape[1], render_func=renderer, fig_dir=FIGURES_DIR
)

In [ ]:
(
    family.res_taskvar["r2test"].nanmedian(),
    family_mb.res_taskvar["r2test"].nanmedian(),
    family_mf.res_taskvar["r2test"].nanmedian(),
)

In [ ]:
plt.figure()
plt.scatter(family.res_taskvar["r2test"], family_mf.res_taskvar["r2test"])
plt.plot([-1, 1], [-1, 1])
plt.show()

In [ ]:
family.cids.shape, family_mb.cids.shape

In [ ]:
plt.figure()
plt.plot([-2, 1], [-2, 1])
plt.scatter(
    family_mf.res_taskvar["r2test"][family.cids],
    family_mb.res_taskvar["r2test"][family.cids],
)
plt.show()

In [ ]:
plt.figure()
plt.scatter(
    family.res_taskvar["r2test"][family.cids],
    family_mb.res_taskvar["r2test"][family.cids],
)
plt.show()

In [ ]:
from utils.paths import MODELS_DIR
import pickle

subj_id = "MR82"
sess_id = "20251027_152036"
file_path = (
    MODELS_DIR / "fit" / subj_id / sess_id / "river_n_tributaries" / "results_dict.pkl"
)

with open(file_path, "rb") as f:
    res_dict = pickle.load(f)

In [ ]:
strategy = "mb"
reg = "all"
metric = "r2test_taskvar"

metrics = []
epochs_key = ["choice", "reward", "iti"]
for epoch in epochs_key:
    if strategy == "both":
        families = res_dict[reg][epoch][strategy]["families"]
        if len(families) == 0:
            # region doesn't exist for this session
            break
    else:
        try:
            families = res_dict[reg][epoch][strategy]["families"]
        except KeyError:
            # region doesn't exist for this session
            break

    for j, family in enumerate(families):
        family.eval()
        if metric == "qi":
            metric_ = family.qi
        elif metric == "r2test_taskvar":
            try:
                metric_ = family.res_taskvar["r2test"].nanmedian()
            except AttributeError:
                metric_ = np.nan
        elif metric == "r2test_affine":
            try:
                metric_ = family.res_affine["r2test"].nanmedian()
            except AttributeError:
                metric_ = np.nan
        metrics.append(metric_)

In [ ]:
metrics

In [ ]:
(
    family.res_taskvar["r2test"].mean(),
    family_mb.res_taskvar["r2test"].mean(),
    family_mf.res_taskvar["r2test"].mean(),
)

In [ ]:
# def plot_beta_tv(family_mb, family_mf, tv, color=False):
#     beta_mb = family_mb.mod_taskvar.tv.weight.data[:]
#     beta_mf = family_mf.mod_taskvar.tv.weight.data[:]

#     tv_idxs = []
#     tv_labels = []
#     counter = 0
#     for tv_ in family_mb.task_vars:
#         for val in family_mb.trial_data[tv_].unique():
#             if tv_ == tv:
#                 tv_idxs.append(counter)
#                 tv_labels.append(f"{tv_}_{val}")
#             counter += 1

#     fig, axes = plt.subplots(nrows=1, ncols=len(tv_idxs))

#     for i, ax in enumerate(axes.flat):
#         if color:
#             c = ax.scatter(
#                 beta_mb[tv_idxs[i]],
#                 beta_mf[tv_idxs[i]],
#                 s=0.5,
#                 cmap="hsv",
#                 c=np.arange(beta_mb.shape[1]),
#             )
#             fig.colorbar(c)
#         else:
#             ax.scatter(beta_mb[tv_idxs[i]], beta_mf[tv_idxs[i]], s=0.5, color="#17612F")
#         ax.set_xlabel(r"mb $\beta$")
#         ax.set_ylabel(r"mf $\beta$")
#         ax.set_title(f"{tv_labels[i]}")
#     fig.tight_layout()

In [ ]:
# # load and check metrics are ok
# subj_id = "MR82"
# sess_id = "20251028_140930"

# file_path = (
#     MODELS_DIR / "fit" / subj_id / sess_id / "river_n_tributaries" / "results_dict.pkl"
# )

# with open(file_path, "rb") as f:
#     res_dict = pickle.load(f)

reg = "all"
strategy = "both"

metrics = {}
for epoch in epochs_key:
    families = res_dict[reg][epoch][strategy]["families"]
    metrics[epoch] = []

    for fam in families:
        metrics[epoch].append(fam.qi)  # fam.res_affine["r2test"].mean())
print(metrics)

# # check that beta weight plots are okay
# family    = res_dict["all"]["choice"]["both"]["families"][0]
# family_mb = res_dict["all"]["choice"]["mb"]["families"][0]
# family_mf = res_dict["all"]["choice"]["mf"]["families"][0]

# plot_beta_tv(family_mb, family_mf, "response", color=False)

# # check the performance of neurons for mb vs mf trials
# # epoch as row and region as column
# plt.figure()
# r2    = family.res_taskvar["r2test"]
# r2_mb = family_mb.res_taskvar["r2test"]
# r2_mf = family_mf.res_taskvar["r2test"]
# plt.scatter(r2, r2_mb)
# # plt.plot([0, 1])
# plt.show()

In [ ]:
epochs = [
    {"key": "choice", "alignment": "choice", "tpre": 0.5, "tpost": 0.5},
    {"key": "reward", "alignment": "reward", "tpre": 0, "tpost": 1},
    {"key": "iti", "alignment": "trial_start", "tpre": 1.5, "tpost": -0.5},
]
epochs_key = [epoch["key"] for epoch in epochs]

n_cv = 5


def get_metrics(subj_id, reg, strategy, metric):
    sess_ids = session_ids[np.where(subject_ids == subj_id)[0][0]]
    metrics = {epoch: np.full((len(sess_ids), n_cv), np.nan) for epoch in epochs_key}

    for i, sess_id in enumerate(sess_ids):
        print(sess_id)
        file_path = (
            MODELS_DIR
            / "fit"
            / subj_id
            / sess_id
            / "river_n_tributaries"
            / "results_dict.pkl"
        )

        if not file_path.is_file():
            continue

        with open(file_path, "rb") as f:
            res_dict = pickle.load(f)

        for epoch in epochs_key:
            print(epoch)
            if strategy == "both":
                families = res_dict[reg][epoch][strategy]["families"]
                if len(families) == 0:
                    # region doesn't exist for this session
                    break
            else:
                try:
                    families = res_dict[reg][epoch][strategy]["families"]
                except KeyError:
                    # region doesn't exist for this session
                    break

            for j, family in enumerate(families):
                if metric == "qi":
                    metric_ = family.qi
                elif metric == "r2test_taskvar":
                    try:
                        metric_ = family.res_taskvar["r2test"].mean()
                    except AttributeError:
                        metric_ = np.nan
                elif metric == "r2test_affine":
                    try:
                        metric_ = family.res_affine["r2test"].mean()
                    except AttributeError:
                        metric_ = np.nan
                metrics[epoch][i][j] = metric_
    return metrics


# metrics = get_metrics("MR82", "all", "both", "r2test_taskvar")

In [ ]:
regions = ["all", "ACC", "M2", "DMS", "DLS"]
strategies = ["both", "mb", "mf"]


def get_metrics_3x3(subj_id, metric):
    metrics = {}
    for reg in regions:
        print(f"! {reg}")
        metrics[reg] = {}
        for strategy in strategies:
            print(f"> {strategy}")
            metrics[reg][strategy] = {}

            metrics[reg][strategy] = get_metrics(subj_id, reg, strategy, metric)
            print(metrics[reg][strategy])

        if np.array(
            [
                np.isnan(metrics[reg][strategy][epoch])
                for epoch in epochs_key
                for strategy in strategies
            ]
        ).all():
            metrics.pop(reg, None)
    return metrics


# metrics = get_metrics_3x3("MR82", "r2test_taskvar")
# from scipy.stats import sem

# colors_epoch = {"choice": "#1F6A92", "reward": "#229B46", "trial_start": "#7051B8"}
# fig, ax = plt.subplots()

# for epoch in epochs_key:
#     session_idx = np.arange(metrics[epoch].shape[0])
#     metrics_mean = np.nanmean(metrics[epoch], axis=1)
#     metrics_sem = sem(metrics[epoch], nan_policy="omit", axis=1)

#     ax.plot(session_idx, metrics_mean, color=colors_epoch[epoch], label=epoch)
#     ax.fill_between(
#         session_idx,
#         metrics_mean - metrics_sem,
#         metrics_mean + metrics_sem,
#         color=colors_epoch[epoch],
#         alpha=0.5,
#     )

# fig.legend()

In [ ]:
from core.data import colors_epoch
from scipy.stats import sem


def plot_metrics_3x3(subj_id, metrics, metric, do_save=True):
    fig, axes = plt.subplots(ncols=len(metrics), nrows=3, figsize=(5, 4))

    for i, reg in enumerate(metrics):
        for j, strategy in enumerate(metrics[reg]):
            for epoch in metrics[reg][strategy]:
                metrics_epoch = metrics[reg][strategy][epoch]
                metrics_avg = np.nanmean(metrics_epoch, axis=1)
                metrics_sem = sem(metrics_epoch, axis=1, nan_policy="omit")

                session_idxs = np.arange(metrics_epoch.shape[0])

                axes[j][i].plot(
                    session_idxs, metrics_avg, color=colors_epoch[epoch], label=epoch
                )
                axes[j][i].fill_between(
                    session_idxs,
                    metrics_avg - metrics_sem,
                    metrics_avg + metrics_sem,
                    color=colors_epoch[epoch],
                    alpha=0.5,
                )
            axes[j][i].set_xlabel("Sessions")
            axes[j][i].set_ylabel(metric)
            axes[j][i].legend()

    fig.tight_layout()
    if do_save:
        from utils.paths import FIGURES_DIR

        fpath_png = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}.png"
        fpath_svg = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}.svg"
        fpath_png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(fpath_png, dpi=300, bbox_inches="tight")
        fig.savefig(fpath_svg, dpi=300, bbox_inches="tight")


subj_id = "MR82"
metric = "r2test_taskvar"

# metrics = get_metrics_3x3(subj_id, metric)
plot_metrics_3x3(subj_id, metrics, metric)

In [ ]:
def plot_metrics_3x3_bar(subj_id, metrics, metric, do_save=True):
    fig, axes = plt.subplots(ncols=len(metrics), nrows=3, figsize=(5, 4))

    for i, reg in enumerate(metrics):
        for j, strategy in enumerate(metrics[reg]):
            epochs = metrics[reg][strategy].keys()
            metrics_avg = [
                np.nanmean(metrics[reg][strategy][epoch])
                for epoch in metrics[reg][strategy]
            ]
            metrics_sem = [
                sem(metrics[reg][strategy][epoch], nan_policy="omit")
                for epoch in metrics[reg][strategy]
            ]

            axes[j][i].bar(
                epochs,
                metrics_avg,
                color=[colors_epoch[epoch] for epoch in metrics[reg][strategy]],
            )
            axes[j][i].errorbar(epochs, metrics_avg, metrics_sem, color="k")
            axes[j][i].set_ylabel(metric)

    fig.tight_layout()
    if do_save:
        from utils.paths import FIGURES_DIR

        fpath_png = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}_bars.png"
        fpath_svg = FIGURES_DIR / "reg_strategy_epoch" / subj_id / f"{metric}_bars.svg"
        fpath_png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(fpath_png, dpi=300, bbox_inches="tight")
        fig.savefig(fpath_svg, dpi=300, bbox_inches="tight")

In [ ]:
plot_metrics_3x3_bar(subj_id, metrics, metric)

In [ ]:
plot_metrics_3x3(subj_id, metrics, metric)

In [ ]:
metrics_taskvar = get_metrics("MR82", "all", "both", metric="r2test_taskvar")

In [ ]:
metrics_baseline = get_metrics("MR82", "all", "both", metric="r2test_baseline")

In [ ]:
epochs = [
    {"alignment": "choice", "tpre": 0.5, "tpost": 0.5},
    {"alignment": "reward", "tpre": 0, "tpost": 1},
    {"alignment": "trial_start", "tpre": 1.5, "tpost": -0.5},
]
metrics_mean = {
    epoch["alignment"]: np.nanmean(metrics_taskvar[epoch["alignment"]], axis=1)
    for epoch in epochs
}
metrics_std = {
    epoch["alignment"]: np.nanstd(metrics_taskvar[epoch["alignment"]], axis=1)
    for epoch in epochs
}

In [ ]:
plt.figure()
for epoch in epochs:
    plt.plot(
        metrics_mean[epoch["alignment"]],
        color=colors_epoch[epoch["alignment"]],
        label=epoch["alignment"],
    )
    plt.fill_between(
        x=np.arange(len(metrics_mean[epoch["alignment"]])),
        y1=np.array(metrics_mean[epoch["alignment"]])
        - np.array(metrics_std[epoch["alignment"]]),
        y2=np.array(metrics_mean[epoch["alignment"]])
        + np.array(metrics_std[epoch["alignment"]]),
        alpha=0.5,
        color=colors_epoch[epoch["alignment"]],
    )
plt.axhline(y=0, color="#626262", linewidth=0.5, linestyle="--")
plt.xlabel("session id")
plt.ylabel(r"$r^2$ taskvar model")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# look at beta weights, make comparisons, reference todo and keep track

In [ ]:
colors_epoch = {"choice": "#1F6A92", "reward": "#229B46", "trial_start": "#7051B8"}
plt.figure()
for epoch in epochs:
    plt.plot(
        metrics_mean[epoch["alignment"]],
        color=colors_epoch[epoch["alignment"]],
        label=epoch["alignment"],
    )
    plt.fill_between(
        x=np.arange(len(metrics_mean[epoch["alignment"]])),
        y1=np.array(metrics_mean[epoch["alignment"]])
        - np.array(metrics_std[epoch["alignment"]]),
        y2=np.array(metrics_mean[epoch["alignment"]])
        + np.array(metrics_std[epoch["alignment"]]),
        alpha=0.5,
        color=colors_epoch[epoch["alignment"]],
    )
plt.axhline(y=0, color="#626262", linewidth=0.5, linestyle="--")
plt.xlabel("session id")
plt.ylabel(r"$r^2$ baseline model")
plt.legend()
plt.tight_layout()
plt.show()

## Encoder - two strategies

In [ ]:
def load_family_strategy(subj_id, sess_id, region, seed):
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "separate_strategy"
        / "encoder_no_update_cid"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        return

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)

    family_mb = res_dict[region]["mb"]["families"][seed]
    family_mf = res_dict[region]["mf"]["families"][seed]

    return family_mb, family_mf

In [ ]:
from sg.fitter import LVMFamily

subj_id = "MR82"
sess_id = "20251027_152036"
reg = "DLS"
seed = 0

family = LVMFamily(
    subj_id=subj_id,
    sess_id=sess_id,
    n_latents_mult=1,
    n_latents_addt=1,
    sanity_check=0,
    task_vars=[
        "response",
        "rewarded",
        "block_side",
        "response_prev",
        "rewarded_prev",
    ],
    n_splines=5,
    tpre=0.5,
    tpost=1,
)
family.fit_all()
family.eval()

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from squiggs.neuron_viewer import NeuronViewer
from core.data import get_psths_cond, get_choice_ts
from utils.paths import FIGURES_DIR
from pathlib import Path


def plot_psths_renderers(family, mode, reg, strategy):
    renderer = PETHRasterRenderer(
        event_times=get_choice_ts(family.trial_data, mode=mode),
        spike_times=family.spike_times[reg],
        peths=get_psths_cond(family.psths[reg], family.trial_data, mode=mode),
        pres=0.5,
        posts=1,
        binwidth_s=25 / 1000,
        s=0.5,
        linewidths=0.5,
        save_subdir=Path("peths") / subj_id / sess_id / reg / mode / strategy,
    )

    _ = NeuronViewer(
        num_units=family.psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
    )

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"
reg = "DLS"
seed = 0

family_mb, family_mf = load_family_strategy(subj_id, sess_id, reg, seed)

In [ ]:
# try ridge and look at weights

In [ ]:
plot_psths_renderers(family, "response", reg, "all")

In [ ]:
plot_psths_renderers(family_mb, "response", reg, "mb")

In [ ]:
plot_psths_renderers(family_mf, "response", reg, "mf")

In [ ]:
# find regl constant fit to full model then fit mb and mf separately on different epochs

In [ ]:
def plot_beta_tv(subj_id, sess_id, tv, region, seed, color=False):
    family_mb, family_mf = load_family_strategy(subj_id, sess_id, region, seed)

    beta_mb = family_mb.mod_taskvar.tv.weight.data[:]
    beta_mf = family_mf.mod_taskvar.tv.weight.data[:]

    tv_idxs = []
    tv_labels = []
    counter = 0
    for tv_ in family_mb.task_vars:
        for val in family_mb.trial_data[tv_].unique():
            if tv_ == tv:
                tv_idxs.append(counter)
                tv_labels.append(f"{tv_}_{val}")
            counter += 1

    fig, axes = plt.subplots(nrows=1, ncols=len(tv_idxs))

    for i, ax in enumerate(axes.flat):
        if color:
            c = ax.scatter(
                beta_mb[tv_idxs[i]],
                beta_mf[tv_idxs[i]],
                s=0.5,
                cmap="hsv",
                c=np.arange(beta_mb.shape[1]),
            )
            fig.colorbar(c)
        else:
            ax.scatter(beta_mb[tv_idxs[i]], beta_mf[tv_idxs[i]], s=0.5, color="#17612F")
        ax.set_xlabel(r"mb $\beta$")
        ax.set_ylabel(r"mf $\beta$")
        ax.set_title(f"{tv_labels[i]}")
    fig.tight_layout()


plot_beta_tv(subj_id, sess_id, "response", reg, seed, color=False)

## LVM - two strategies, one latent

In [ ]:
import numpy as np


def get_res_tv_lvms_strategy(subj_id, regions=["all", "ACC", "M2", "DMS", "DLS"]):
    subj_idx = np.where(subject_ids == subj_id)[0][0]

    res_tv_lvms = []
    n_trials = []
    for sess_id in session_ids[subj_idx]:
        file_path = (
            MODELS_DIR
            / "fit"
            / subj_id
            / sess_id
            / "separate_strategy"
            / "results_dict.pkl"
        )

        if not file_path.is_file():
            continue

        with open(file_path, "rb") as f:
            res_dict = pickle.load(f)

        res_tv_lvm = {"mb": {}, "mf": {}}
        try:
            n_trials.append(res_dict["all"]["mb"]["families"][0].num_trials)
        except KeyError:
            continue

        for region in regions:
            try:
                res_tv_lvms_mb_ = res_dict[region]["mb"]["res_tv_lvms"]
                res_tv_lvms_mf_ = res_dict[region]["mf"]["res_tv_lvms"]
            except KeyError:
                continue
            # for now
            if len(res_tv_lvms_mb_) == 0 or len(res_tv_lvms_mf_) == 0:
                continue
            print(subj_id, sess_id, region)
            res_tv_lvm["mb"][region] = {
                key: np.array([res_tv_lvm_[key] for res_tv_lvm_ in res_tv_lvms_mb_])
                for key in res_tv_lvms_mb_[0].keys()
            }
            res_tv_lvm["mf"][region] = {
                key: np.array([res_tv_lvm_[key] for res_tv_lvm_ in res_tv_lvms_mf_])
                for key in res_tv_lvms_mf_[0].keys()
            }
        res_tv_lvms.append(res_tv_lvm)

    return res_tv_lvms, n_trials

In [ ]:
out = {
    subj_id: get_res_tv_lvms_strategy(subj_id) for subj_id in ["MM012", "MR82", "MR83"]
}

res_tv_lvms_strategy = {
    subj_id: out[subj_id][0] for subj_id in ["MM012", "MR82", "MR83"]
}
n_trials = {subj_id: out[subj_id][1] for subj_id in ["MM012", "MR82", "MR83"]}

In [ ]:
from core.data import colors_strategy

region = "all"
metric = "r2test_affine"
subj_id = "MR83"

fig, ax = plt.subplots()

metrics_avg_mb = np.array(
    [
        res_tv_lvm["mb"][region][metric].mean()
        if region in res_tv_lvm["mb"].keys()
        else np.nan
        for res_tv_lvm in res_tv_lvms_strategy[subj_id]
    ]
)
metrics_std_mb = np.array(
    [
        res_tv_lvm["mb"][region][metric].std()
        if region in res_tv_lvm["mb"].keys()
        else np.nan
        for res_tv_lvm in res_tv_lvms_strategy[subj_id]
    ]
)

metrics_avg_mf = np.array(
    [
        res_tv_lvm["mf"][region][metric].mean()
        if region in res_tv_lvm["mf"].keys()
        else np.nan
        for res_tv_lvm in res_tv_lvms_strategy[subj_id]
    ]
)
metrics_std_mf = np.array(
    [
        res_tv_lvm["mf"][region][metric].std()
        if region in res_tv_lvm["mf"].keys()
        else np.nan
        for res_tv_lvm in res_tv_lvms_strategy[subj_id]
    ]
)

ax.plot(metrics_avg_mb, color=colors_strategy["mb"], label="mb")
ax.plot(metrics_avg_mf, color=colors_strategy["mf"], label="mf")
ax.fill_between(
    np.arange(len(metrics_avg_mb)),
    metrics_avg_mb - metrics_std_mb,
    metrics_avg_mb + metrics_std_mb,
    color=colors_strategy["mb"],
    alpha=0.25,
)
ax.fill_between(
    np.arange(len(metrics_avg_mf)),
    metrics_avg_mf - metrics_std_mf,
    metrics_avg_mf + metrics_std_mf,
    color=colors_strategy["mf"],
    alpha=0.25,
)

ax2 = ax.twinx()
ax2.plot(n_trials[subj_id], color="#888888")
ax2.set_ylabel("n. trials")

ax.axhline(y=0, color="#666666", linestyle="--")
ax.set_xlabel("session")
ax.set_ylabel(metric)
ax.legend()
fig.tight_layout()


save_dir = FIGURES_DIR / "session_summary" / "separate_strategy"
fpath = save_dir / f"{metric}_{region}.png"
fpath.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(fpath, dpi=300, bbox_inches="tight")

In [ ]:
def load_family_strategy(subj_id, sess_id, region, seed):
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "separate_strategy"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        return

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)

    family_mb = res_dict[region]["mb"]["families"][seed]
    family_mf = res_dict[region]["mf"]["families"][seed]

    return family_mb, family_mf

In [ ]:
family_mb, family_mf = load_family_strategy("MR82", "20251027_152036", "all", 0)

In [ ]:
(
    family_mb.mod_taskvar.tv.weight.data[:].shape,
    family_mf.mod_taskvar.tv.weight.data[:].shape,
)

In [ ]:
from sg.eval_models import get_coupling


def plot_coupling_strategy(subj_id, sess_id, region, seed):
    file_path = (
        MODELS_DIR
        / "fit"
        / subj_id
        / sess_id
        / "separate_strategy"
        / "results_dict.pkl"
    )

    if not file_path.is_file():
        return

    with open(file_path, "rb") as f:
        res_dict = pickle.load(f)

    family_mb = res_dict[region]["mb"]["families"][seed]
    family_mf = res_dict[region]["mf"]["families"][seed]

    fig, axes = plt.subplots(nrows=2, ncols=2)

    # gain
    coupling_gg_mb = get_coupling(family_mb.mod_gain, mode="gain")
    coupling_gg_mf = get_coupling(family_mf.mod_gain, mode="gain")

    # offset
    coupling_oo_mb = get_coupling(family_mb.mod_offset, mode="offset")
    coupling_oo_mf = get_coupling(family_mf.mod_offset, mode="offset")

    # affine
    coupling_ga_mb = get_coupling(family_mb.mod_affine, mode="gain")
    coupling_oa_mb = get_coupling(family_mb.mod_affine, mode="offset")
    coupling_ga_mf = get_coupling(family_mf.mod_affine, mode="gain")
    coupling_oa_mf = get_coupling(family_mf.mod_affine, mode="offset")

    # PLOT
    print(coupling_gg_mb.shape, coupling_gg_mf.shape)
    axes[0, 0].scatter(coupling_gg_mb, coupling_gg_mf)
    axes[0, 1].scatter(coupling_oo_mb, coupling_oo_mf)
    axes[1, 0].scatter(coupling_ga_mb, coupling_ga_mf)
    axes[1, 1].scatter(coupling_oa_mb, coupling_oa_mf)

In [ ]:
plot_coupling_strategy("MR82", "20251027_152036", "all", 0)